<a href="https://colab.research.google.com/github/eodenyire/LearnDataScienceWithEmmanuelOdenyire/blob/main/Phase_5_Specialization/Track_4_Computer_Vision/transfer_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transfer Learning with Pretrained Models
## Learning Objectives
- Understand transfer learning concepts (feature extraction vs. fine-tuning)
- Apply ResNet/EfficientNet to custom image tasks
- Implement fine-tuning with frozen and unfrozen layers


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, Model

# Load pretrained EfficientNetB0 (ImageNet weights)
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_model.trainable = False  # freeze base layers

print(f'Total layers in EfficientNetB0: {len(base_model.layers)}')
print(f'Trainable parameters: {base_model.count_params():,}')

In [ ]:
# Add custom classification head
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(5, activation='softmax')(x)  # 5-class example

model = Model(inputs=base_model.input, outputs=outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])

total = sum(1 for l in model.layers if l.trainable)
print(f'Trainable layers (head only): {total}')
print('Phase 1: Train head with frozen base')

In [ ]:
# Fine-tuning: unfreeze top layers
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 30  # unfreeze last 30 layers
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # much lower LR
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
total_trainable = sum(1 for l in model.layers if l.trainable)
print(f'Trainable layers after fine-tuning: {total_trainable}')
print('Phase 2: Fine-tune top layers with very low learning rate')

In [ ]:
# Transfer learning strategies
strategies = {
    'Feature Extraction': 'Freeze all pretrained layers, train only new head. Fast, needs little data.',
    'Fine-Tuning': 'Unfreeze top N layers and train with tiny LR. Better accuracy, needs more data.',
    'Full Retraining': 'Unfreeze all layers. Best accuracy but needs lots of data and compute.',
}

print('Transfer Learning Strategies:')
for strategy, desc in strategies.items():
    print(f'  {strategy:20s}: {desc}')

print('\nData size guidelines:')
for size, rec in [('<1K', 'Feature extraction only'), ('1K-10K', 'Fine-tune top layers'), ('>10K', 'Full fine-tuning')]:
    print(f'  {size:8s}: {rec}')

## Exercises
1. Download a flower classification dataset and fine-tune EfficientNetB0.
2. Compare training from scratch vs. transfer learning on 500 images.
3. Visualize activation maps using Grad-CAM.


## Summary
- Pretrained models provide powerful feature extractors for free.
- Feature extraction is fastest; fine-tuning gives better accuracy.
- Use a very low learning rate when fine-tuning pretrained weights.
